In [ ]:
import os
import numpy as np
import pandas as pd

BASE_DIR = r"C:\REPLACE_WITH_LOCAL_DATA\reliability_experiments"

folders = {
    "CNN": "cnn_baseline",
    "Transformer-only": "transformer_only",
    "Capsule-only": "capsule_only",
    "Capsule+Transformer": "capsule_transformer"
}

print("Checking experiment folders...\n")

for model_name, folder_name in folders.items():
    folder = os.path.join(BASE_DIR, folder_name)

    y_true_path = os.path.join(folder, "y_true.npy")
    y_prob_path = os.path.join(folder, "y_probability.npy")

    print(model_name)
    print("  Folder exists:", os.path.isdir(folder))
    print("  y_true exists:", os.path.isfile(y_true_path))
    print("  y_probability exists:", os.path.isfile(y_prob_path))

    if os.path.isfile(y_true_path) and os.path.isfile(y_prob_path):
        y_true = np.load(y_true_path)
        y_prob = np.load(y_prob_path)

        print("  y_true shape:", y_true.shape)
        print("  y_probability shape:", y_prob.shape)

    print()

In [ ]:
# ============================================================
# VERIFY SAME TEST LABELS ACROSS ALL FOUR MODELS
# ============================================================

all_y_true = {}

for model_name, folder_name in folders.items():
    path = os.path.join(
        BASE_DIR,
        folder_name,
        "y_true.npy"
    )

    all_y_true[model_name] = np.load(path)

reference_name = "CNN"
reference_y = all_y_true[reference_name]

print("=" * 60)
print("TEST LABEL CONSISTENCY CHECK")
print("=" * 60)

for model_name, y in all_y_true.items():

    same_shape = reference_y.shape == y.shape
    exact_match = np.array_equal(reference_y, y)

    print(f"{model_name:22s}")
    print("  Same shape :", same_shape)
    print("  Exact labels/order match CNN :", exact_match)

print("\nReference test samples:", len(reference_y))
print("Class 0 (PAL):", np.sum(reference_y == 0))
print("Class 1 (PER):", np.sum(reference_y == 1))

# Strong safety assertion
for model_name, y in all_y_true.items():
    assert np.array_equal(reference_y, y), \
        f"ERROR: Test labels/order differ for {model_name}"

print("\n✅ ALL FOUR MODELS USE IDENTICAL TEST LABELS AND ORDER")

In [ ]:
# ============================================================
# CALIBRATION METRICS
# ECE + BRIER SCORE + NEGATIVE LOG-LIKELIHOOD
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss, log_loss


def expected_calibration_error(y_true, y_prob, n_bins=10):
    """
    Standard equal-width Expected Calibration Error (ECE).
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    # Predicted class and confidence
    y_pred = (y_prob >= 0.5).astype(int)
    confidence = np.where(
        y_pred == 1,
        y_prob,
        1.0 - y_prob
    )

    correctness = (y_pred == y_true).astype(float)

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)

    ece = 0.0

    for i in range(n_bins):
        lower = bin_edges[i]
        upper = bin_edges[i + 1]

        if i == 0:
            mask = (confidence >= lower) & (confidence <= upper)
        else:
            mask = (confidence > lower) & (confidence <= upper)

        if np.any(mask):
            bin_accuracy = np.mean(correctness[mask])
            bin_confidence = np.mean(confidence[mask])
            bin_weight = np.mean(mask)

            ece += bin_weight * abs(
                bin_accuracy - bin_confidence
            )

    return ece


results = []

for model_name, folder_name in folders.items():

    y_true = np.load(
        os.path.join(
            BASE_DIR,
            folder_name,
            "y_true.npy"
        )
    )

    y_prob = np.load(
        os.path.join(
            BASE_DIR,
            folder_name,
            "y_probability.npy"
        )
    )

    # Numerical safety for log loss
    y_prob_safe = np.clip(
        y_prob,
        1e-7,
        1.0 - 1e-7
    )

    ece = expected_calibration_error(
        y_true,
        y_prob,
        n_bins=10
    )

    brier = brier_score_loss(
        y_true,
        y_prob
    )

    nll = log_loss(
        y_true,
        y_prob_safe
    )

    results.append({
        "Model": model_name,
        "ECE": ece,
        "Brier Score": brier,
        "NLL": nll
    })


calibration_results = pd.DataFrame(results)

print("=" * 70)
print("CALIBRATION / RELIABILITY RESULTS")
print("=" * 70)

print(
    calibration_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nLower is better for ECE, Brier Score, and NLL.")

In [ ]:
# ============================================================
# RELIABILITY DIAGRAMS — UNCALIBRATED TEST PREDICTIONS
# ============================================================

import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

SAVE_DIR = os.path.join(
    BASE_DIR,
    "calibration_analysis"
)

os.makedirs(SAVE_DIR, exist_ok=True)

for model_name, folder_name in folders.items():

    y_true = np.load(
        os.path.join(
            BASE_DIR,
            folder_name,
            "y_true.npy"
        )
    )

    y_prob = np.load(
        os.path.join(
            BASE_DIR,
            folder_name,
            "y_probability.npy"
        )
    )

    prob_true, prob_pred = calibration_curve(
        y_true,
        y_prob,
        n_bins=10,
        strategy="uniform"
    )

    plt.figure(figsize=(6, 6))

    # Perfect calibration reference
    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="Perfect calibration"
    )

    # Model calibration
    plt.plot(
        prob_pred,
        prob_true,
        marker="o",
        label=model_name
    )

    plt.xlabel("Mean Predicted Probability")
    plt.ylabel("Fraction of Positives")
    plt.title(f"Reliability Diagram — {model_name}")

    plt.xlim(0, 1)
    plt.ylim(0, 1)

    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()

    safe_name = (
        model_name.lower()
        .replace("+", "_plus_")
        .replace("-", "_")
        .replace(" ", "_")
    )

    plt.savefig(
        os.path.join(
            SAVE_DIR,
            f"reliability_{safe_name}.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()


# Save numerical calibration table too
calibration_results.to_csv(
    os.path.join(
        SAVE_DIR,
        "uncalibrated_calibration_metrics.csv"
    ),
    index=False
)

print("\n✅ RELIABILITY DIAGRAMS SAVED")
print("Folder:")
print(SAVE_DIR)

print("\nSaved files:")
for f in sorted(os.listdir(SAVE_DIR)):
    print(f)

In [ ]:
# ============================================================
# BINARY PROBABILITY ECE
# CONSISTENT WITH RELIABILITY DIAGRAM
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss, log_loss


def binary_ece(y_true, y_prob, n_bins=10):
    """
    Binary ECE:
    bins predicted probability P(class=1),
    then compares mean predicted probability
    with observed fraction of class 1.
    """

    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)

    ece = 0.0

    for i in range(n_bins):

        lower = bin_edges[i]
        upper = bin_edges[i + 1]

        if i == 0:
            mask = (y_prob >= lower) & (y_prob <= upper)
        else:
            mask = (y_prob > lower) & (y_prob <= upper)

        n_bin = np.sum(mask)

        if n_bin > 0:

            mean_probability = np.mean(y_prob[mask])
            observed_fraction = np.mean(y_true[mask])

            weight = n_bin / len(y_true)

            ece += weight * abs(
                observed_fraction - mean_probability
            )

    return ece


corrected_results = []

for model_name, folder_name in folders.items():

    y_true = np.load(
        os.path.join(
            BASE_DIR,
            folder_name,
            "y_true.npy"
        )
    )

    y_prob = np.load(
        os.path.join(
            BASE_DIR,
            folder_name,
            "y_probability.npy"
        )
    )

    y_prob_safe = np.clip(
        y_prob,
        1e-7,
        1 - 1e-7
    )

    ece = binary_ece(
        y_true,
        y_prob,
        n_bins=10
    )

    brier = brier_score_loss(
        y_true,
        y_prob
    )

    nll = log_loss(
        y_true,
        y_prob_safe
    )

    corrected_results.append({
        "Model": model_name,
        "ECE": ece,
        "Brier Score": brier,
        "NLL": nll
    })


corrected_calibration_results = pd.DataFrame(
    corrected_results
)

print("=" * 72)
print("CORRECTED BINARY CALIBRATION RESULTS")
print("ECE NOW MATCHES THE RELIABILITY-DIAGRAM DEFINITION")
print("=" * 72)

print(
    corrected_calibration_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nLower is better for all three metrics.")


# Save corrected table
corrected_calibration_results.to_csv(
    os.path.join(
        SAVE_DIR,
        "binary_calibration_metrics_corrected.csv"
    ),
    index=False
)

print("\n✅ Corrected calibration table saved")

In [ ]:
# ============================================================
# CHECK FOR SAVED VALIDATION PREDICTIONS
# ============================================================

print("=" * 65)
print("CHECKING VALIDATION PREDICTIONS")
print("=" * 65)

for model_name, folder_name in folders.items():

    folder = os.path.join(BASE_DIR, folder_name)

    possible_files = [
        "validation_predictions.csv",
        "val_predictions.csv",
        "y_val_probability.npy",
        "validation_probability.npy",
        "val_probability.npy"
    ]

    print(f"\n{model_name}")

    found = False

    for filename in possible_files:
        path = os.path.join(folder, filename)

        if os.path.isfile(path):
            print("  FOUND:", filename)
            found = True

    if not found:
        print("  No saved validation probabilities found.")

In [ ]:
# ============================================================
# RECREATE VALIDATION DATASET FROM SAVED MANIFEST
# ============================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf

BASE_DIR = r"C:\REPLACE_WITH_LOCAL_DATA\reliability_experiments"

# Use CNN validation manifest as reference
VAL_MANIFEST = os.path.join(
    BASE_DIR,
    "cnn_baseline",
    "validation_manifest.csv"
)

val_df = pd.read_csv(VAL_MANIFEST)

print("Validation rows:", len(val_df))
print("Validation subjects:", val_df["subject"].nunique())
print(val_df["modality"].value_counts())

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def decode_img(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=1)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    img = tf.ensure_shape(img, [224, 224, 1])
    return img

def make_val_ds(frame):
    paths = frame["path"].values
    labels = frame["label"].values.astype(np.int32)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    ds = ds.map(
        lambda p, y: (decode_img(p), y),
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

    return ds

val_ds = make_val_ds(val_df)

print(
    "Validation batches:",
    tf.data.experimental.cardinality(val_ds).numpy()
)

for x_batch, y_batch in val_ds.take(1):
    print("Image batch shape:", x_batch.shape)
    print("Label batch shape:", y_batch.shape)

In [ ]:
# ============================================================
# CNN VALIDATION DATASET — 3 CHANNEL RGB
# ============================================================

def decode_img_rgb(path):
    img = tf.io.read_file(path)

    # Decode directly as RGB because saved CNN expects 3 channels
    img = tf.image.decode_png(img, channels=3)

    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0

    img = tf.ensure_shape(img, [224, 224, 3])

    return img


def make_val_ds_rgb(frame):
    paths = frame["path"].values
    labels = frame["label"].values.astype(np.int32)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    ds = ds.map(
        lambda p, y: (decode_img_rgb(p), y),
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

    return ds


cnn_val_ds = make_val_ds_rgb(val_df)

for x_batch, y_batch in cnn_val_ds.take(1):
    print("CNN image batch shape:", x_batch.shape)
    print("CNN label batch shape:", y_batch.shape)

print(
    "CNN validation batches:",
    tf.data.experimental.cardinality(cnn_val_ds).numpy()
)

In [ ]:
# ============================================================
# GENERATE VALIDATION PROBABILITIES — CNN
# CORRECT VERSION: USES 3-CHANNEL cnn_val_ds
# ============================================================

import tensorflow as tf
import numpy as np
import os

CNN_DIR = os.path.join(
    BASE_DIR,
    "cnn_baseline"
)

cnn_model_path = os.path.join(
    CNN_DIR,
    "cnn_baseline_model.keras"
)

print("Loading CNN model:")
print(cnn_model_path)

cnn_model = tf.keras.models.load_model(
    cnn_model_path,
    compile=False
)

print("\nCNN model loaded successfully.")

print("\nGenerating validation probabilities...")

cnn_val_probability = cnn_model.predict(
    cnn_val_ds,
    verbose=1
).ravel()

cnn_val_true = np.concatenate([
    y.numpy() for _, y in cnn_val_ds
])

print("\nValidation samples:", len(cnn_val_true))
print("Probability shape:", cnn_val_probability.shape)
print("Labels shape:", cnn_val_true.shape)

print("PAL:", np.sum(cnn_val_true == 0))
print("PER:", np.sum(cnn_val_true == 1))

np.save(
    os.path.join(
        CNN_DIR,
        "y_val_probability.npy"
    ),
    cnn_val_probability
)

np.save(
    os.path.join(
        CNN_DIR,
        "y_val_true.npy"
    ),
    cnn_val_true
)

print("\n✅ CNN VALIDATION PREDICTIONS SAVED")

In [ ]:
# ============================================================
# Experiment 1 — Transformer-Only Baseline
# Same train/validation/test sets as Capsule–Transformer
# ============================================================

import tensorflow as tf
from tensorflow.keras import layers, Model

# Reproducibility
SEED = 42
tf.keras.utils.set_random_seed(SEED)

class TransformerOnlyBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.2):
        super().__init__()

        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dropout(dropout),
            layers.Dense(embed_dim)
        ])

        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def call(self, x, training=False):

        attn = self.attention(
            query=x,
            value=x,
            key=x,
            training=training
        )

        attn = self.dropout1(attn, training=training)
        x = self.norm1(x + attn)

        ffn = self.ffn(x, training=training)
        ffn = self.dropout2(ffn, training=training)

        return self.norm2(x + ffn)


def build_transformer_only_model(
    input_shape=(224, 224, 1),
    token_dim=64,
    num_heads=4,
    ff_dim=128,
    transformer_layers=2,
    dropout=0.2
):

    inp = layers.Input(shape=input_shape)

    # Convert grayscale spectrogram to RGB for MobileNetV2
    x = layers.Lambda(
        lambda z: tf.image.grayscale_to_rgb(z),
        name="gray_to_rgb"
    )(inp)

    # Same frozen backbone used in Capsule–Transformer
    base = tf.keras.applications.MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=(224, 224, 3)
    )

    base.trainable = False
    x = base(x, training=False)

    # --------------------------------------------------------
    # Transformer tokens directly from CNN feature map
    # NO capsule representation is used here
    # --------------------------------------------------------

    # MobileNetV2 output ≈ 7 × 7 × 1280
    x = layers.Conv2D(
        token_dim,
        kernel_size=1,
        padding="same",
        name="token_projection"
    )(x)

    # 7 × 7 spatial locations -> 49 Transformer tokens
    x = layers.Reshape(
        (49, token_dim),
        name="transformer_tokens"
    )(x)

    x = layers.LayerNormalization(
        epsilon=1e-6,
        name="token_norm"
    )(x)

    # Transformer encoder
    for i in range(transformer_layers):
        x = TransformerOnlyBlock(
            embed_dim=token_dim,
            num_heads=num_heads,
            ff_dim=ff_dim,
            dropout=dropout
        )(x)

    # Global token aggregation
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(dropout)(x)

    output = layers.Dense(
        1,
        activation="sigmoid"
    )(x)

    return Model(
        inp,
        output,
        name="TransformerOnly"
    )


transformer_model = build_transformer_only_model()

transformer_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="acc"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

transformer_model.summary()

In [ ]:
# ============================================================
# LOAD SAVED TRANSFORMER-ONLY MODEL
# ============================================================

TRANS_DIR = os.path.join(
    BASE_DIR,
    "transformer_only"
)

trans_model_path = os.path.join(
    TRANS_DIR,
    "transformer_only_model.keras"
)

print("Loading Transformer-only model:")
print(trans_model_path)

trans_model = tf.keras.models.load_model(
    trans_model_path,
    custom_objects={
        "TransformerOnlyBlock": TransformerOnlyBlock
    },
    compile=False
)

print("\n✅ Transformer-only model loaded successfully.")
print("Input shape:", trans_model.input_shape)
print("Output shape:", trans_model.output_shape)

In [ ]:
# ============================================================
# GENERATE VALIDATION PROBABILITIES — TRANSFORMER-ONLY
# ============================================================

print("Generating Transformer-only validation probabilities...")

trans_val_probability = trans_model.predict(
    val_ds,
    verbose=1
).ravel()

trans_val_true = np.concatenate([
    y.numpy() for _, y in val_ds
])

print("\nValidation samples:", len(trans_val_true))
print("Probability shape:", trans_val_probability.shape)
print("Labels shape:", trans_val_true.shape)

print("PAL:", np.sum(trans_val_true == 0))
print("PER:", np.sum(trans_val_true == 1))

np.save(
    os.path.join(
        TRANS_DIR,
        "y_val_probability.npy"
    ),
    trans_val_probability
)

np.save(
    os.path.join(
        TRANS_DIR,
        "y_val_true.npy"
    ),
    trans_val_true
)

print("\n✅ TRANSFORMER-ONLY VALIDATION PREDICTIONS SAVED")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K

# -------------------------
# Capsule functions
# -------------------------

def squash(vectors, axis=-1):
    s_squared_norm = K.sum(K.square(vectors), axis, keepdims=True)
    scale = s_squared_norm / (1 + s_squared_norm) / K.sqrt(s_squared_norm + K.epsilon())
    return scale * vectors

# -------------------------
# Build Capsule Model
# -------------------------

def build_capsule_model(input_shape=(224, 224, 1)):
    inputs = layers.Input(shape=input_shape)

    # Grayscale → RGB
    x = layers.Lambda(lambda x: tf.image.grayscale_to_rgb(x))(inputs)

    # CNN backbone
    base = tf.keras.applications.MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights="imagenet"
    )
    base.trainable = False
    x = base(x)

    # -------- Primary Capsules --------
    # Conv → reshape → squash
    x = layers.Conv2D(
        filters=8 * 16,
        kernel_size=3,
        strides=2,
        padding="valid",
        activation="relu"
    )(x)

    # Explicit reshape (this fixes the None issue)
    x = layers.Reshape((-1, 16))(x)
    x = layers.Lambda(squash)(x)

    # -------- Digit Capsules (2 classes) --------
    # Simple routing-free version (stable & fast)
    x = layers.Dense(16, activation="relu")(x)
    x = layers.GlobalAveragePooling1D()(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = models.Model(inputs, outputs)
    return model

# -------------------------
# Compile
# -------------------------

capsule_model = build_capsule_model()

capsule_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="acc"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

capsule_model.summary()

In [ ]:
# ============================================================
# LOAD TRAINED CAPSULE-ONLY WEIGHTS
# ============================================================

CAPS_DIR = os.path.join(
    BASE_DIR,
    "capsule_only"
)

caps_model_path = os.path.join(
    CAPS_DIR,
    "capsule_only_model.keras"
)

print("Loading trained Capsule-only weights from:")
print(caps_model_path)

capsule_model.load_weights(
    caps_model_path
)

print("\n✅ CAPSULE-ONLY TRAINED WEIGHTS LOADED")
print("Input shape:", capsule_model.input_shape)
print("Output shape:", capsule_model.output_shape)

In [ ]:
# ============================================================
# GENERATE VALIDATION PROBABILITIES — CAPSULE-ONLY
# ============================================================

print("Generating Capsule-only validation probabilities...")

caps_val_probability = capsule_model.predict(
    val_ds,
    verbose=1
).ravel()

caps_val_true = np.concatenate([
    y.numpy() for _, y in val_ds
])

print("\nValidation samples:", len(caps_val_true))
print("Probability shape:", caps_val_probability.shape)
print("Labels shape:", caps_val_true.shape)

print("PAL:", np.sum(caps_val_true == 0))
print("PER:", np.sum(caps_val_true == 1))

np.save(
    os.path.join(
        CAPS_DIR,
        "y_val_probability.npy"
    ),
    caps_val_probability
)

np.save(
    os.path.join(
        CAPS_DIR,
        "y_val_true.npy"
    ),
    caps_val_true
)

print("\n✅ CAPSULE-ONLY VALIDATION PREDICTIONS SAVED")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

# -------------------------
# Transformer block
# -------------------------
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()

        self.att = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])

        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)

        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, inputs, training=False):

        attn_output = self.att(
            inputs,
            inputs,
            training=training
        )

        attn_output = self.drop1(
            attn_output,
            training=training
        )

        out1 = self.layernorm1(
            inputs + attn_output
        )

        ffn_output = self.ffn(
            out1,
            training=training
        )

        ffn_output = self.drop2(
            ffn_output,
            training=training
        )

        return self.layernorm2(
            out1 + ffn_output
        )

In [ ]:
# ============================================================
# LOAD SAVED CAPSULE + TRANSFORMER MODEL
# ============================================================

CAPS_TRANS_DIR = os.path.join(
    BASE_DIR,
    "capsule_transformer"
)

caps_trans_model_path = os.path.join(
    CAPS_TRANS_DIR,
    "capsule_transformer_model.keras"
)

print("Loading Capsule+Transformer model:")
print(caps_trans_model_path)

caps_trans_model = tf.keras.models.load_model(
    caps_trans_model_path,
    custom_objects={
        "TransformerBlock": TransformerBlock
    },
    compile=False
)

print("\n✅ Capsule+Transformer model loaded successfully.")
print("Input shape:", caps_trans_model.input_shape)
print("Output shape:", caps_trans_model.output_shape)

In [ ]:
# ============================================================
# GENERATE VALIDATION PROBABILITIES — CAPSULE + TRANSFORMER
# ============================================================

print("Generating Capsule+Transformer validation probabilities...")

caps_trans_val_probability = caps_trans_model.predict(
    val_ds,
    verbose=1
).ravel()

caps_trans_val_true = np.concatenate([
    y.numpy() for _, y in val_ds
])

print("\nValidation samples:", len(caps_trans_val_true))
print("Probability shape:", caps_trans_val_probability.shape)
print("Labels shape:", caps_trans_val_true.shape)

print("PAL:", np.sum(caps_trans_val_true == 0))
print("PER:", np.sum(caps_trans_val_true == 1))

np.save(
    os.path.join(
        CAPS_TRANS_DIR,
        "y_val_probability.npy"
    ),
    caps_trans_val_probability
)

np.save(
    os.path.join(
        CAPS_TRANS_DIR,
        "y_val_true.npy"
    ),
    caps_trans_val_true
)

print("\n✅ CAPSULE+TRANSFORMER VALIDATION PREDICTIONS SAVED")

In [ ]:
# ============================================================
# VERIFY VALIDATION LABELS — ALL FOUR MODELS
# ============================================================

val_true_all = {}

for model_name, folder in folders.items():

    path = os.path.join(
        BASE_DIR,
        folder,
        "y_val_true.npy"
    )

    y = np.load(path)
    val_true_all[model_name] = y

    print(
        model_name,
        "| shape:", y.shape,
        "| PAL:", np.sum(y == 0),
        "| PER:", np.sum(y == 1)
    )

reference = val_true_all["CNN"]

print("\nExact validation-label/order comparison with CNN:")

all_same = True

for model_name, y in val_true_all.items():

    same = np.array_equal(
        reference,
        y
    )

    print(
        model_name,
        "=", same
    )

    if not same:
        all_same = False

if all_same:
    print(
        "\n✅ ALL FOUR MODELS USE IDENTICAL VALIDATION LABELS AND ORDER"
    )
else:
    print(
        "\n❌ VALIDATION LABEL/ORDER MISMATCH DETECTED"
    )

In [ ]:
# ============================================================
# FIT TEMPERATURE SCALING ON VALIDATION SET ONLY
# ============================================================

import numpy as np
import pandas as pd
import os
from scipy.optimize import minimize_scalar

def probability_to_logit(p, eps=1e-7):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def binary_nll(y_true, y_prob, eps=1e-7):
    y_prob = np.clip(y_prob, eps, 1 - eps)

    return -np.mean(
        y_true * np.log(y_prob)
        +
        (1 - y_true) * np.log(1 - y_prob)
    )

temperature_results = []

for model_name, folder in folders.items():

    model_dir = os.path.join(
        BASE_DIR,
        folder
    )

    y_val = np.load(
        os.path.join(
            model_dir,
            "y_val_true.npy"
        )
    )

    p_val = np.load(
        os.path.join(
            model_dir,
            "y_val_probability.npy"
        )
    )

    logits = probability_to_logit(p_val)

    nll_before = binary_nll(
        y_val,
        p_val
    )

    def objective(T):

        calibrated_prob = sigmoid(
            logits / T
        )

        return binary_nll(
            y_val,
            calibrated_prob
        )

    result = minimize_scalar(
        objective,
        bounds=(0.05, 10.0),
        method="bounded"
    )

    best_T = result.x

    calibrated_val_prob = sigmoid(
        logits / best_T
    )

    nll_after = binary_nll(
        y_val,
        calibrated_val_prob
    )

    temperature_results.append({
        "Model": model_name,
        "Temperature": best_T,
        "Validation_NLL_Before": nll_before,
        "Validation_NLL_After": nll_after
    })

    np.save(
        os.path.join(
            model_dir,
            "y_val_probability_temperature_scaled.npy"
        ),
        calibrated_val_prob
    )

temperature_df = pd.DataFrame(
    temperature_results
)

print(temperature_df.to_string(index=False))

CALIB_DIR = os.path.join(
    BASE_DIR,
    "calibration_analysis"
)

os.makedirs(
    CALIB_DIR,
    exist_ok=True
)

temperature_df.to_csv(
    os.path.join(
        CALIB_DIR,
        "temperature_scaling_validation_results.csv"
    ),
    index=False
)

print(
    "\n✅ TEMPERATURE SCALING FITTED USING VALIDATION DATA ONLY"
)

In [ ]:
# ============================================================
# APPLY VALIDATION-FITTED TEMPERATURE TO UNTOUCHED TEST SET
# ============================================================

test_calibration_results = []

for _, row in temperature_df.iterrows():

    model_name = row["Model"]
    best_T = row["Temperature"]

    folder = folders[model_name]

    model_dir = os.path.join(
        BASE_DIR,
        folder
    )

    # Load original untouched test data
    y_test = np.load(
        os.path.join(
            model_dir,
            "y_true.npy"
        )
    )

    p_test = np.load(
        os.path.join(
            model_dir,
            "y_probability.npy"
        )
    )

    # Convert original probabilities to logits
    test_logits = probability_to_logit(
        p_test
    )

    # Apply temperature learned ONLY from validation data
    p_test_calibrated = sigmoid(
        test_logits / best_T
    )

    # NLL before and after
    nll_before = binary_nll(
        y_test,
        p_test
    )

    nll_after = binary_nll(
        y_test,
        p_test_calibrated
    )

    test_calibration_results.append({
        "Model": model_name,
        "Temperature_from_Validation": best_T,
        "Test_NLL_Before": nll_before,
        "Test_NLL_After": nll_after
    })

    # Save calibrated test probabilities
    np.save(
        os.path.join(
            model_dir,
            "y_probability_temperature_scaled.npy"
        ),
        p_test_calibrated
    )


test_temperature_df = pd.DataFrame(
    test_calibration_results
)

print(
    test_temperature_df.to_string(index=False)
)

test_temperature_df.to_csv(
    os.path.join(
        CALIB_DIR,
        "temperature_scaling_test_results.csv"
    ),
    index=False
)

print(
    "\n✅ VALIDATION-FITTED TEMPERATURES APPLIED TO UNTOUCHED TEST SET"
)

In [ ]:
# ============================================================
# FINAL TEST CALIBRATION METRICS
# BEFORE vs AFTER TEMPERATURE SCALING
# ============================================================

from sklearn.metrics import brier_score_loss

def binary_ece(y_true, y_prob, n_bins=10):

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)

    ece = 0.0

    for i in range(n_bins):

        if i == n_bins - 1:
            mask = (
                (y_prob >= bin_edges[i]) &
                (y_prob <= bin_edges[i + 1])
            )
        else:
            mask = (
                (y_prob >= bin_edges[i]) &
                (y_prob < bin_edges[i + 1])
            )

        if np.any(mask):

            mean_probability = np.mean(
                y_prob[mask]
            )

            observed_fraction = np.mean(
                y_true[mask]
            )

            weight = np.sum(mask) / len(y_true)

            ece += weight * abs(
                observed_fraction -
                mean_probability
            )

    return ece


final_results = []

for _, row in temperature_df.iterrows():

    model_name = row["Model"]
    folder = folders[model_name]

    model_dir = os.path.join(
        BASE_DIR,
        folder
    )

    y_test = np.load(
        os.path.join(
            model_dir,
            "y_true.npy"
        )
    )

    p_before = np.load(
        os.path.join(
            model_dir,
            "y_probability.npy"
        )
    )

    p_after = np.load(
        os.path.join(
            model_dir,
            "y_probability_temperature_scaled.npy"
        )
    )

    final_results.append({

        "Model": model_name,

        "ECE_Before":
            binary_ece(y_test, p_before),

        "ECE_After":
            binary_ece(y_test, p_after),

        "Brier_Before":
            brier_score_loss(y_test, p_before),

        "Brier_After":
            brier_score_loss(y_test, p_after),

        "NLL_Before":
            binary_nll(y_test, p_before),

        "NLL_After":
            binary_nll(y_test, p_after)
    })


final_calibration_df = pd.DataFrame(
    final_results
)

print(
    final_calibration_df.to_string(index=False)
)

final_calibration_df.to_csv(
    os.path.join(
        CALIB_DIR,
        "final_test_calibration_metrics.csv"
    ),
    index=False
)

print(
    "\n✅ FINAL TEST CALIBRATION METRICS SAVED"
)

In [ ]:
# ============================================================
# FINAL RELIABILITY DIAGRAMS
# BEFORE vs AFTER TEMPERATURE SCALING
# ============================================================

import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
import os
import numpy as np

for model_name, folder in folders.items():

    model_dir = os.path.join(
        BASE_DIR,
        folder
    )

    y_test = np.load(
        os.path.join(
            model_dir,
            "y_true.npy"
        )
    )

    p_before = np.load(
        os.path.join(
            model_dir,
            "y_probability.npy"
        )
    )

    p_after = np.load(
        os.path.join(
            model_dir,
            "y_probability_temperature_scaled.npy"
        )
    )

    # Reliability curve before calibration
    prob_true_before, prob_pred_before = calibration_curve(
        y_test,
        p_before,
        n_bins=10,
        strategy="uniform"
    )

    # Reliability curve after calibration
    prob_true_after, prob_pred_after = calibration_curve(
        y_test,
        p_after,
        n_bins=10,
        strategy="uniform"
    )

    plt.figure(figsize=(7, 6))

    # Perfect calibration line
    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="Perfect calibration"
    )

    # Before
    plt.plot(
        prob_pred_before,
        prob_true_before,
        marker="o",
        label="Before temperature scaling"
    )

    # After
    plt.plot(
        prob_pred_after,
        prob_true_after,
        marker="s",
        label="After temperature scaling"
    )

    plt.xlabel("Mean Predicted Probability")
    plt.ylabel("Observed Fraction of Positive Class")

    plt.title(
        f"Reliability Diagram — {model_name}"
    )

    plt.xlim(0, 1)
    plt.ylim(0, 1)

    plt.grid(alpha=0.3)
    plt.legend()

    plt.tight_layout()

    safe_name = (
        model_name
        .lower()
        .replace("+", "_plus_")
        .replace("-", "_")
        .replace(" ", "_")
    )

    save_path = os.path.join(
        CALIB_DIR,
        f"reliability_before_after_{safe_name}.png"
    )

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print(
        "Saved:",
        save_path
    )

print(
    "\n✅ ALL FINAL BEFORE/AFTER RELIABILITY DIAGRAMS SAVED"
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
from pathlib import Path

# Final held-out test-set confusion matrices
confusion_matrices = {
    "CNN": np.array([[1391, 49],
                     [22, 1418]]),

    "Transformer-only": np.array([[1368, 72],
                                  [1, 1439]]),

    "Capsule-only": np.array([[1349, 91],
                              [0, 1440]]),

    "Capsule–Transformer": np.array([[1364, 76],
                                     [0, 1440]])
}

fig, axes = plt.subplots(2, 2, figsize=(11, 9))

panel_labels = ["(a)", "(b)", "(c)", "(d)"]

for ax, (model_name, cm), panel in zip(
        axes.ravel(), confusion_matrices.items(), panel_labels):

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Percussion", "Palpation"]
    )

    disp.plot(
        ax=ax,
        cmap="Blues",
        colorbar=False,
        values_format="d"
    )

    ax.set_title(f"{panel} {model_name}", fontsize=13, fontweight="bold")
    ax.set_xlabel("Predicted Class", fontsize=11)
    ax.set_ylabel("True Class", fontsize=11)

fig.tight_layout()

# Save in the same calibration-analysis folder
out_path = Path(
    r"C:\REPLACE_WITH_LOCAL_DATA\reliability_experiments\calibration_analysis"
) / "fig_results_confusion_matrices_17sep.png"

plt.savefig(
    out_path,
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print("Saved to:")
print(out_path)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# Final Sensors Figure — Capsule–Transformer Training Curves
# Uses ACTUAL saved training history
# ============================================================

history_path = Path(
    r"C:\REPLACE_WITH_LOCAL_DATA\reliability_experiments\capsule_transformer\training_history.csv"
)

df = pd.read_csv(history_path)

epochs = range(1, len(df) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ------------------------------------------------------------
# (a) Binary cross-entropy loss
# ------------------------------------------------------------
axes[0].plot(
    epochs,
    df["loss"],
    marker="o",
    linewidth=2,
    label="Training"
)

axes[0].plot(
    epochs,
    df["val_loss"],
    marker="s",
    linewidth=2,
    label="Validation"
)

axes[0].set_title("(a) Loss", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Epoch", fontsize=12)
axes[0].set_ylabel("Binary Cross-Entropy Loss", fontsize=12)
axes[0].set_xticks(list(epochs))
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=True)


# ------------------------------------------------------------
# (b) ROC–AUC
# ------------------------------------------------------------
axes[1].plot(
    epochs,
    df["auc"],
    marker="o",
    linewidth=2,
    label="Training"
)

axes[1].plot(
    epochs,
    df["val_auc"],
    marker="s",
    linewidth=2,
    label="Validation"
)

axes[1].set_title("(b) ROC–AUC", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Epoch", fontsize=12)
axes[1].set_ylabel("ROC–AUC", fontsize=12)
axes[1].set_xticks(list(epochs))
axes[1].grid(alpha=0.25)
axes[1].legend(frameon=True)

fig.tight_layout()

# ------------------------------------------------------------
# Save publication-quality image
# ------------------------------------------------------------
output_path = Path(
    r"C:\REPLACE_WITH_LOCAL_DATA\reliability_experiments\calibration_analysis"
) / "Figure_Training_Capsule_Transformer.png"

plt.savefig(
    output_path,
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print("Saved to:")
print(output_path)